# Hyperparameter Tuning - Assignment

Welcome to the assignment for Hyperparameter Tuning.

Hyperparameter tuning is a crucial step in building effective machine learning models. While model parameters are learned automatically from data, hyperparameters control how the learning process behaves and must be chosen before training. These include settings such as learning rate, regularization strength, number of estimators, or model complexity. Selecting appropriate hyperparameters can significantly improve model performance, stability, and generalization.

In this assignment, you will explore different strategies for optimizing hyperparameters, from simple exhaustive searches to more advanced probabilistic approaches. You will learn how each method explores the search space, what tradeoffs it introduces, and when it should be preferred in practice.

Hyperparameter tuning methods aim to find configurations that maximize model performance while balancing computational cost. Some strategies guarantee optimal solutions but are expensive, while others trade completeness for efficiency. Understanding these differences is essential for building scalable and reliable machine learning systems.

**What You Will Do in This Assignment**

* Understand the role of hyperparameters and how they differ from model parameters.
* Implement Grid Search to exhaustively explore hyperparameter combinations.
* Apply Random Search for more efficient exploration of large search spaces.
* Use Bayesian Optimization to guide the search using probabilistic modeling.
* Analyze hyperparameter importance to identify the most influential parameters.
* Compare tuning strategies in terms of performance and computational efficiency.
* Build a Grid Search algorithm from scratch using cross-validation.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

1. [Introduction to Hyperparameter Tuning](#1)
2. [Grid Search](#2)
   - [Exercise 1: Implement Grid Search](#ex-1)
3. [Random Search](#3)
   - [Exercise 2: Implement Random Search](#ex-2)
4. [Bayesian Optimization](#4)
   - [Exercise 3: Implement Bayesian Optimization](#ex-3)
5. [Hyperparameter Importance](#5)
   - [Exercise 4: Analyze Parameter Importance](#ex-4)
6. [Strategy Comparison](#6)
   - [Exercise 5: Compare Tuning Methods](#ex-5)
7. [From-Scratch Implementation](#7)
   - [Exercise 6: Grid Search from Scratch](#ex-6)
8. [Conclusion](#8)

<a name='1'></a>
## 1 - Introduction to Hyperparameter Tuning

**Hyperparameters** are configuration settings that control the learning process but are not learned from data. Finding optimal hyperparameters is crucial for model performance.

### Parameters vs Hyperparameters:

**Parameters** (learned from data):
- Neural network weights
- Linear regression coefficients
- Decision tree split points

**Hyperparameters** (set before training):
- Learning rate
- Number of layers/neurons
- Regularization strength
- Max tree depth
- Batch size

### Search Space:

The **search space** $\Theta$ defines all possible hyperparameter configurations:
$$\Theta = \theta_1 \times \theta_2 \times ... \times \theta_n$$

where each $\theta_i$ represents the valid range for hyperparameter $i$.

### Objective Function:

We aim to find:
$$\theta^* = \arg\max_{\theta \in \Theta} f(\theta)$$

where $f(\theta)$ is typically cross-validation score.

### Tuning Strategies:

1. **Manual Search**: Trial and error based on experience
2. **Grid Search**: Exhaustive search over discrete values
3. **Random Search**: Random sampling from distributions
4. **Bayesian Optimization**: Model-based sequential search
5. **Evolutionary Algorithms**: Population-based optimization
6. **Gradient-Based**: For differentiable hyperparameters

### Key Concepts:

- **Overfitting in Tuning**: Selecting parameters that overfit validation set
- **Computational Budget**: Time/resources available for search
- **Exploration vs Exploitation**: Breadth vs depth in search
- **Early Stopping**: Terminating poor configurations quickly

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import unittests
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.datasets import make_classification, load_digits
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.preprocessing import StandardScaler

# Bayesian optimization
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Warning: optuna not installed. Install with: pip install optuna")

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")

### Helper Functions

In [ ]:
def load_dataset():
    """Load and prepare dataset for hyperparameter tuning."""
    # Load digits dataset (simpler than full MNIST)
    digits = load_digits()
    X, y = digits.data, digits.target
    
    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test


def plot_search_results(results_df, title="Hyperparameter Search Results"):
    """Plot hyperparameter search results."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Score progression
    axes[0].plot(results_df.index, results_df['mean_score'], 
                 marker='o', linewidth=2, markersize=6)
    axes[0].fill_between(results_df.index, 
                         results_df['mean_score'] - results_df['std_score'],
                         results_df['mean_score'] + results_df['std_score'],
                         alpha=0.2)
    axes[0].axhline(y=results_df['mean_score'].max(), color='red', 
                    linestyle='--', label=f'Best: {results_df["mean_score"].max():.4f}')
    axes[0].set_xlabel('Iteration', fontsize=12)
    axes[0].set_ylabel('Mean CV Score', fontsize=12)
    axes[0].set_title('Score Progression', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Time per iteration
    if 'time' in results_df.columns:
        axes[1].bar(results_df.index, results_df['time'], alpha=0.7, color='green')
        axes[1].set_xlabel('Iteration', fontsize=12)
        axes[1].set_ylabel('Time (seconds)', fontsize=12)
        axes[1].set_title('Time per Iteration', fontsize=12, fontweight='bold')
        axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def plot_parameter_importance(param_scores, title="Parameter Importance"):
    """Plot importance of different hyperparameters."""
    plt.figure(figsize=(10, 6))
    params = list(param_scores.keys())
    scores = list(param_scores.values())
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(params)))
    bars = plt.barh(params, scores, color=colors, alpha=0.8)
    
    plt.xlabel('Importance Score', fontsize=12)
    plt.ylabel('Hyperparameter', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()


def compare_methods(methods_results):
    """Compare different tuning methods."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    methods = list(methods_results.keys())
    best_scores = [methods_results[m]['best_score'] for m in methods]
    total_times = [methods_results[m]['total_time'] for m in methods]
    n_iterations = [methods_results[m]['n_iterations'] for m in methods]
    
    # Best scores
    axes[0].bar(methods, best_scores, color='steelblue', alpha=0.8)
    axes[0].set_ylabel('Best Score', fontsize=12)
    axes[0].set_title('Best Score Comparison', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Total time
    axes[1].bar(methods, total_times, color='coral', alpha=0.8)
    axes[1].set_ylabel('Total Time (seconds)', fontsize=12)
    axes[1].set_title('Computation Time', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Efficiency (score per second)
    efficiency = [best_scores[i] / total_times[i] for i in range(len(methods))]
    axes[2].bar(methods, efficiency, color='mediumseagreen', alpha=0.8)
    axes[2].set_ylabel('Score per Second', fontsize=12)
    axes[2].set_title('Efficiency', fontsize=12, fontweight='bold')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

print("Helper functions defined!")

### Load Dataset

In [ ]:
# Load and prepare data
X_train, X_test, y_train, y_test = load_dataset()

print("Dataset loaded successfully!")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")
print(f"Feature dimension: {X_train.shape[1]}")

<a name='2'></a>
## 2 - Grid Search

**Grid Search** performs exhaustive search over a manually specified subset of the hyperparameter space.

### Algorithm:

For each combination of hyperparameters:
1. Train model with current configuration
2. Evaluate using cross-validation
3. Record performance score
4. Return best configuration

### Mathematical Formulation:

Given parameter grids $\theta_1 = \{v_1^1, v_2^1, ..., v_{n_1}^1\}$, $\theta_2 = \{v_1^2, v_2^2, ..., v_{n_2}^2\}$, etc.

Total configurations: $N = n_1 \times n_2 \times ... \times n_k$

For each configuration $\theta_i$:
$$\text{score}(\theta_i) = \frac{1}{K} \sum_{j=1}^{K} \text{eval}(\mathcal{M}_{\theta_i}, \mathcal{D}_j)$$

where $K$ is number of folds, $\mathcal{M}_{\theta_i}$ is model with parameters $\theta_i$, and $\mathcal{D}_j$ is fold $j$.

### Advantages:
- Guaranteed to find best combination in grid
- Straightforward to implement and understand
- Easy to parallelize
- Reproducible results

### Disadvantages:
- Exponential growth with number of parameters
- Wastes computation on poor regions
- Must discretize continuous parameters
- Doesn't adapt based on results

### When to Use:
- Small number of hyperparameters (< 4)
- Limited parameter ranges
- Need to understand parameter interactions
- Sufficient computational budget

<a name='ex-1'></a>
### Exercise 1 - Implement Grid Search

**Your Task:**

Implement the `perform_grid_search` function that:
1. Defines a parameter grid for Random Forest
2. Performs exhaustive grid search with cross-validation
3. Tracks all configurations and their scores
4. Returns best parameters and detailed results

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For parameter grid:**
* Define dictionary: `param_grid = {'n_estimators': [50, 100, 200], ...}`
* Common RF params: n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features

**For grid search:**
* Create searcher: `grid_search = GridSearchCV(model, param_grid, cv=cv_folds, scoring=scoring)`
* Fit: `grid_search.fit(X_train, y_train)`
* Best params: `grid_search.best_params_`
* Best score: `grid_search.best_score_`
* All results: `grid_search.cv_results_`

**For results analysis:**
* Extract scores: `cv_results_['mean_test_score']`
* Extract std: `cv_results_['std_test_score']`
* Extract params: `cv_results_['params']`

</details>

In [ ]:
# GRADED FUNCTION: perform_grid_search
def perform_grid_search(X_train, y_train, cv_folds=3, verbose=True):
    """
    Perform grid search for Random Forest hyperparameters.
    
    Args:
        X_train: Training features
        y_train: Training labels
        cv_folds: Number of cross-validation folds
        verbose: Whether to print progress
    
    Returns:
        Dictionary with best_params, best_score, results_df, total_time
    """
    ### START CODE HERE ### (≈ 20 lines)
    
    # Define parameter grid
    param_grid = None
    
    # Create Random Forest classifier
    rf = None
    
    # Create GridSearchCV object
    grid_search = None
    
    # Perform grid search
    start_time = None
    # Fit here
    end_time = None
    
    total_time = None
    
    # Extract results
    best_params = None
    best_score = None
    
    # Create results dataframe
    results_df = None
    
    ### END CODE HERE ###
    
    if verbose:
        print(f"Grid Search completed in {total_time:.2f} seconds")
        print(f"Best Score: {best_score:.4f}")
        print(f"Best Parameters: {best_params}")
    
    return {
        'best_params': best_params,
        'best_score': best_score,
        'results_df': results_df,
        'total_time': total_time,
        'n_iterations': len(results_df)
    }

In [ ]:
# Perform grid search
print("="*60)
print("GRID SEARCH")
print("="*60)

grid_results = perform_grid_search(X_train, y_train, cv_folds=3, verbose=True)

print(f"\nTotal configurations evaluated: {grid_results['n_iterations']}")
print(f"\nTop 5 configurations:")
print(grid_results['results_df'].head())

# Visualize results
plot_search_results(grid_results['results_df'], title="Grid Search Results")

##### **Expected Output**
```
============================================================
GRID SEARCH
============================================================
Grid Search completed in XX.XX seconds
Best Score: 0.9XXX
Best Parameters: {'max_depth': XX, 'max_features': 'XXX', ...}

Total configurations evaluated: XXX

Top 5 configurations:
   mean_score  std_score  ...
0     0.9XXX    0.0XXX   ...
...
```
You should see multiple configurations evaluated with varying performance. The best configuration should have the highest mean score.

In [ ]:
unittests.exercise_1(perform_grid_search)

<a name='3'></a>
## 3 - Random Search

**Random Search** samples random combinations from parameter distributions rather than exhaustively trying all combinations.

### Key Insight:

Not all hyperparameters are equally important. Random search can find good configurations faster by:
- Exploring more values for important parameters
- Not wasting time on complete grids

### Algorithm:

1. Define parameter distributions
2. For $n$ iterations:
   - Sample random configuration
   - Train and evaluate model
   - Track performance
3. Return best configuration found

### Mathematical Formulation:

Sample $\theta \sim \mathcal{P}(\Theta)$ where $\mathcal{P}$ is the distribution over hyperparameters.

Common distributions:
- **Uniform**: $\theta \sim \mathcal{U}(a, b)$
- **Log-uniform**: $\log(\theta) \sim \mathcal{U}(\log(a), \log(b))$ for learning rates
- **Discrete uniform**: Equal probability for discrete choices
- **Normal**: $\theta \sim \mathcal{N}(\mu, \sigma^2)$

### Advantages:
- More efficient than grid search
- Can handle continuous parameters naturally
- Better exploration of parameter space
- Easy to add more iterations
- Less sensitive to irrelevant parameters

### Disadvantages:
- No guarantee of finding optimal
- May miss important interactions
- Results vary between runs
- Harder to analyze parameter effects

### Bergstra & Bengio (2012) Result:

Random search is more efficient than grid search when:
- Some parameters have low importance
- Parameters have different importance

For the same budget, random search tests more distinct values for important parameters.

<a name='ex-2'></a>
### Exercise 2 - Implement Random Search

**Your Task:**

Implement the `perform_random_search` function that:
1. Defines parameter distributions for Random Forest
2. Randomly samples configurations
3. Evaluates each with cross-validation
4. Compares efficiency with grid search

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For parameter distributions:**
* Integers: Use lists `[10, 20, 50, 100]`
* Continuous: Use `scipy.stats` distributions or `np.random`
* Log-scale: For learning rates, use `loguniform(1e-4, 1e-1)`

**For random search:**
* Create searcher: `random_search = RandomizedSearchCV(model, param_distributions, n_iter=n_iter, cv=cv, scoring=scoring, random_state=42)`
* Fit: `random_search.fit(X_train, y_train)`
* Extract results same as grid search

**For comparison:**
* Track: number of configurations, time, best score
* Compare: score per second, score per iteration

</details>

In [ ]:
# GRADED FUNCTION: perform_random_search
def perform_random_search(X_train, y_train, n_iter=50, cv_folds=3, verbose=True):
    """
    Perform random search for Random Forest hyperparameters.
    
    Args:
        X_train: Training features
        y_train: Training labels
        n_iter: Number of random configurations to try
        cv_folds: Number of cross-validation folds
        verbose: Whether to print progress
    
    Returns:
        Dictionary with best_params, best_score, results_df, total_time
    """
    ### START CODE HERE ### (≈ 20 lines)
    
    # Define parameter distributions
    param_distributions = None
    
    # Create Random Forest classifier
    rf = None
    
    # Create RandomizedSearchCV object
    random_search = None
    
    # Perform random search
    start_time = None
    # Fit here
    end_time = None
    
    total_time = None
    
    # Extract results
    best_params = None
    best_score = None
    
    # Create results dataframe
    results_df = None
    
    ### END CODE HERE ###
    
    if verbose:
        print(f"Random Search completed in {total_time:.2f} seconds")
        print(f"Best Score: {best_score:.4f}")
        print(f"Best Parameters: {best_params}")
    
    return {
        'best_params': best_params,
        'best_score': best_score,
        'results_df': results_df,
        'total_time': total_time,
        'n_iterations': n_iter
    }

In [ ]:
# Perform random search
print("="*60)
print("RANDOM SEARCH")
print("="*60)

random_results = perform_random_search(X_train, y_train, n_iter=50, cv_folds=3, verbose=True)

print(f"\nTotal configurations evaluated: {random_results['n_iterations']}")
print(f"\nTop 5 configurations:")
print(random_results['results_df'].head())

# Visualize results
plot_search_results(random_results['results_df'], title="Random Search Results")

# Compare with grid search
print("\n" + "="*60)
print("GRID SEARCH vs RANDOM SEARCH")
print("="*60)
print(f"Grid Search:   Best={grid_results['best_score']:.4f}, Time={grid_results['total_time']:.2f}s, Configs={grid_results['n_iterations']}")
print(f"Random Search: Best={random_results['best_score']:.4f}, Time={random_results['total_time']:.2f}s, Configs={random_results['n_iterations']}")
print(f"\nEfficiency (score/second):")
print(f"Grid Search:   {grid_results['best_score']/grid_results['total_time']:.6f}")
print(f"Random Search: {random_results['best_score']/random_results['total_time']:.6f}")

##### **Expected Output**
```
============================================================
RANDOM SEARCH
============================================================
Random Search completed in XX.XX seconds
Best Score: 0.9XXX
Best Parameters: {'max_depth': XX, 'max_features': 'XXX', ...}

Total configurations evaluated: 50

============================================================
GRID SEARCH vs RANDOM SEARCH
============================================================
Grid Search:   Best=0.9XXX, Time=XX.XXs, Configs=XXX
Random Search: Best=0.9XXX, Time=XX.XXs, Configs=50

Efficiency (score/second):
Grid Search:   0.0XXXXX
Random Search: 0.0XXXXX
```
Random search typically finds competitive solutions faster than grid search, especially when the number of configurations is limited.

In [ ]:
unittests.exercise_2(perform_random_search)

<a name='4'></a>
## 4 - Bayesian Optimization

**Bayesian Optimization** uses probabilistic models to intelligently select which hyperparameters to try next based on past results.

### Key Idea:

Build a **surrogate model** of the objective function and use it to decide where to sample next.

### Algorithm:

1. **Initialize**: Evaluate a few random configurations
2. **Fit surrogate model**: Model $f(\theta)$ using Gaussian Process or Tree-based model
3. **Acquisition function**: Decide where to sample next
   - Exploitation: Sample where model predicts high performance
   - Exploration: Sample where uncertainty is high
4. **Evaluate**: Train model with selected $\theta$
5. **Update**: Add result to history and repeat

### Surrogate Models:

1. **Gaussian Process (GP)**:
   $$f(\theta) \sim \mathcal{GP}(\mu(\theta), k(\theta, \theta'))$$
   - Provides uncertainty estimates
   - Works well in low dimensions

2. **Tree-based Models** (TPE):
   - Used by Optuna and Hyperopt
   - Better for high dimensions
   - Handles categorical parameters

### Acquisition Functions:

1. **Expected Improvement (EI)**:
   $$EI(\theta) = \mathbb{E}[\max(0, f(\theta) - f(\theta^*))$$

2. **Upper Confidence Bound (UCB)**:
   $$UCB(\theta) = \mu(\theta) + \kappa \sigma(\theta)$$
   
   where $\kappa$ controls exploration-exploitation tradeoff

3. **Probability of Improvement (PI)**:
   $$PI(\theta) = P(f(\theta) > f(\theta^*) + \xi)$$

### Advantages:
- Sample efficient (fewer evaluations needed)
- Handles expensive objective functions
- Balances exploration and exploitation
- Works with continuous and discrete parameters
- Can optimize multiple objectives

### Disadvantages:
- More complex to implement
- Harder to parallelize
- Overhead for simple problems
- Can get stuck in local optima

<a name='ex-3'></a>
### Exercise 3 - Implement Bayesian Optimization

**Your Task:**

Implement the `perform_bayesian_optimization` function using Optuna that:
1. Defines an objective function for hyperparameter optimization
2. Uses TPE sampler for intelligent search
3. Tracks optimization progress
4. Compares with random/grid search

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For objective function:**
* Define function that takes `trial` parameter
* Suggest params: `trial.suggest_int('n_estimators', 50, 200)`
* For categorical: `trial.suggest_categorical('max_features', ['sqrt', 'log2'])`
* Train model and return score

**For Optuna study:**
* Create study: `study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))`
* Optimize: `study.optimize(objective, n_trials=n_trials)`
* Best params: `study.best_params`
* Best value: `study.best_value`
* All trials: `study.trials`

**For results tracking:**
* Extract from each trial: `trial.value`, `trial.params`, `trial.datetime_complete - trial.datetime_start`

</details>

In [ ]:
# GRADED FUNCTION: perform_bayesian_optimization
def perform_bayesian_optimization(X_train, y_train, n_trials=50, cv_folds=3, verbose=True):
    """
    Perform Bayesian optimization for Random Forest hyperparameters using Optuna.
    
    Args:
        X_train: Training features
        y_train: Training labels
        n_trials: Number of trials
        cv_folds: Number of cross-validation folds
        verbose: Whether to print progress
    
    Returns:
        Dictionary with best_params, best_score, results_df, total_time
    """
    if not OPTUNA_AVAILABLE:
        print("Optuna not available. Please install: pip install optuna")
        return None
    
    ### START CODE HERE ### (≈ 25 lines)
    
    # Define objective function
    def objective(trial):
        # Suggest hyperparameters
        params = None  # Define dict with trial.suggest_* methods
        
        # Create model
        rf = None
        
        # Evaluate with cross-validation
        score = None
        
        return None  # Return mean score
    
    # Create study
    study = None
    
    # Optimize
    start_time = None
    # study.optimize here
    end_time = None
    
    total_time = None
    
    # Extract results
    best_params = None
    best_score = None
    
    # Create results dataframe from trials
    results_df = None
    
    ### END CODE HERE ###
    
    if verbose:
        print(f"Bayesian Optimization completed in {total_time:.2f} seconds")
        print(f"Best Score: {best_score:.4f}")
        print(f"Best Parameters: {best_params}")
    
    return {
        'best_params': best_params,
        'best_score': best_score,
        'results_df': results_df,
        'total_time': total_time,
        'n_iterations': n_trials,
        'study': study
    }

In [ ]:
# Perform Bayesian optimization
if OPTUNA_AVAILABLE:
    print("="*60)
    print("BAYESIAN OPTIMIZATION (Optuna)")
    print("="*60)
    
    bayes_results = perform_bayesian_optimization(X_train, y_train, n_trials=50, cv_folds=3, verbose=True)
    
    print(f"\nTotal trials: {bayes_results['n_iterations']}")
    print(f"\nTop 5 trials:")
    print(bayes_results['results_df'].head())
    
    # Visualize results
    plot_search_results(bayes_results['results_df'], title="Bayesian Optimization Results")
    
    # Optuna-specific visualizations
    try:
        from optuna.visualization import plot_optimization_history, plot_param_importances
        
        # Optimization history
        fig = plot_optimization_history(bayes_results['study'])
        fig.show()
        
        # Parameter importances
        fig = plot_param_importances(bayes_results['study'])
        fig.show()
    except:
        print("Optuna visualization not available")
else:
    print("Optuna not installed. Skipping Bayesian optimization.")
    print("Install with: pip install optuna")

##### **Expected Output**
```
============================================================
BAYESIAN OPTIMIZATION (Optuna)
============================================================
Bayesian Optimization completed in XX.XX seconds
Best Score: 0.9XXX
Best Parameters: {'max_depth': XX, 'max_features': 'XXX', ...}

Total trials: 50

Top 5 trials:
   mean_score  std_score  ...
0     0.9XXX    0.0XXX   ...
...
```
Bayesian optimization should converge to good solutions faster than random search, showing improvement in early trials.

In [ ]:
unittests.exercise_3(perform_bayesian_optimization)

<a name='5'></a>
## 5 - Hyperparameter Importance Analysis

Not all hyperparameters equally impact model performance. Understanding which parameters matter most helps focus tuning efforts.

### Why Analyze Importance?

- **Efficiency**: Focus on parameters that matter
- **Understanding**: Learn what drives performance
- **Debugging**: Identify unexpected sensitivities
- **Communication**: Explain model behavior to stakeholders

### Methods:

1. **Functional ANOVA (fANOVA)**:
   - Decomposes variance in performance
   - Attributes variance to each parameter
   - Captures interactions

2. **Ablation Analysis**:
   - Fix one parameter, vary others
   - Measure performance change
   - Quantifies individual impact

3. **Correlation Analysis**:
   - Compute correlation between parameter values and scores
   - Simple but may miss interactions

4. **Feature Importance from Surrogate**:
   - Train model to predict performance from parameters
   - Extract feature importances
   - Available in Optuna and similar tools

### Interpretation:

**High Importance**: Small changes significantly affect performance
- Need careful tuning
- Use finer granularity in search

**Low Importance**: Large changes have minimal effect
- Can use coarse search or defaults
- Less critical to optimize

**Interactions**: Some parameters only matter in combination
- Need to tune jointly
- Grid search better captures interactions

<a name='ex-4'></a>
### Exercise 4 - Analyze Parameter Importance

**Your Task:**

Implement the `analyze_parameter_importance` function that:
1. Takes results from hyperparameter search
2. Computes correlation between each parameter and performance
3. Performs ablation study
4. Visualizes parameter importance

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For correlation analysis:**
* Extract parameters and scores from results
* For numerical params: `correlation = np.abs(np.corrcoef(param_values, scores)[0, 1])`
* For categorical: Use ANOVA or group mean differences

**For ablation study:**
* For each parameter:
  - Fix at best value
  - Vary all others
  - Measure performance drop
* Importance = variance in performance when parameter varies

**For visualization:**
* Bar plot of importance scores
* Sort by importance
* Annotate with actual values

</details>

In [ ]:
# GRADED FUNCTION: analyze_parameter_importance
def analyze_parameter_importance(results_df, method='correlation'):
    """
    Analyze importance of different hyperparameters.
    
    Args:
        results_df: DataFrame with parameters and scores
        method: 'correlation' or 'variance'
    
    Returns:
        Dictionary mapping parameter names to importance scores
    """
    ### START CODE HERE ### (≈ 20 lines)
    
    importance_scores = {}
    
    # Get parameter columns (exclude score/time columns)
    param_cols = None
    
    if method == 'correlation':
        # Compute absolute correlation with score
        for param in param_cols:
            # Handle numerical parameters
            if None:  # check if numeric
                corr = None  # compute correlation
                importance_scores[param] = None
            else:
                # For categorical, use variance of group means
                group_means = None
                importance_scores[param] = None
    
    elif method == 'variance':
        # Compute variance in scores for each parameter value
        for param in param_cols:
            # Group by parameter value and compute score variance
            variance = None
            importance_scores[param] = None
    
    ### END CODE HERE ###
    
    # Sort by importance
    importance_scores = dict(sorted(importance_scores.items(), 
                                   key=lambda x: x[1], reverse=True))
    
    return importance_scores

In [ ]:
# Analyze parameter importance from grid search results
print("="*60)
print("PARAMETER IMPORTANCE ANALYSIS")
print("="*60)

importance = analyze_parameter_importance(grid_results['results_df'], method='correlation')

print("\nParameter Importance (sorted):")
for param, score in importance.items():
    print(f"{param:20s}: {score:.4f}")

# Visualize
plot_parameter_importance(importance, title="Hyperparameter Importance (Correlation Method)")

# Optuna's built-in importance (if available)
if OPTUNA_AVAILABLE and 'bayes_results' in locals():
    print("\n" + "="*60)
    print("OPTUNA PARAMETER IMPORTANCE (fANOVA)")
    print("="*60)
    
    optuna_importance = optuna.importance.get_param_importances(bayes_results['study'])
    print("\nParameter Importance (fANOVA):")
    for param, score in optuna_importance.items():
        print(f"{param:20s}: {score:.4f}")
    
    plot_parameter_importance(optuna_importance, title="Hyperparameter Importance (Optuna fANOVA)")

##### **Expected Output**
```
============================================================
PARAMETER IMPORTANCE ANALYSIS
============================================================

Parameter Importance (sorted):
max_depth           : 0.XXXX
n_estimators        : 0.XXXX
min_samples_split   : 0.XXXX
max_features        : 0.XXXX
...
```
Parameters with higher importance scores have stronger correlation with model performance. Typically, `max_depth` and `n_estimators` are most important for Random Forests.

In [ ]:
unittests.exercise_4(analyze_parameter_importance)

<a name='6'></a>
## 6 - Tuning Strategy Comparison

Different tuning strategies have different strengths. The best choice depends on:
- Computational budget
- Number of hyperparameters
- Training time per configuration
- Required performance level

### Time-Performance Tradeoff:

**Grid Search**:
- Slowest: $O(n_1 \times n_2 \times ... \times n_k)$ evaluations
- Thorough: Guaranteed to find best in grid
- Best for: 2-3 parameters, understanding interactions

**Random Search**:
- Faster: Fixed number of evaluations
- Flexible: Easy to add more trials
- Best for: Many parameters, quick results

**Bayesian Optimization**:
- Most efficient: Fewer evaluations for same quality
- Smart: Learns from past trials
- Best for: Expensive training, limited budget

### Convergence Speed:

Typical behavior:
- **Random**: Linear improvement, plateaus slowly
- **Grid**: Depends on grid design, can be erratic
- **Bayesian**: Fast initial improvement, efficient convergence

### Practical Recommendations:

1. **Start simple**: Try defaults first
2. **Random search**: Quick first pass (50-100 configs)
3. **Analyze importance**: Focus on key parameters
4. **Refined search**: Grid or Bayesian on important params
5. **Ensemble**: Combine multiple good configurations

<a name='ex-5'></a>
### Exercise 5 - Compare Tuning Strategies

**Your Task:**

Implement the `compare_tuning_strategies` function that:
1. Compares all three methods on same dataset
2. Tracks time, best score, and convergence
3. Computes efficiency metrics
4. Provides recommendations

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For comparison:**
* Run all three methods with similar budgets
* Track: best_score, total_time, n_iterations
* Compute: score_per_second, score_per_iteration

**For convergence:**
* Plot best score so far vs iteration
* Use cumulative maximum: `np.maximum.accumulate(scores)`

**For recommendations:**
* If time-constrained: Recommend fastest to good score
* If quality-focused: Recommend best final score
* If balanced: Recommend best efficiency

</details>

In [ ]:
# GRADED FUNCTION: compare_tuning_strategies
def compare_tuning_strategies(X_train, y_train, n_iter_budget=50):
    """
    Compare different hyperparameter tuning strategies.
    
    Args:
        X_train: Training features
        y_train: Training labels
        n_iter_budget: Maximum iterations for each method
    
    Returns:
        Dictionary with comparison results
    """
    ### START CODE HERE ### (≈ 15 lines)
    
    results = {}
    
    # Grid search (might use fewer iterations if grid is small)
    grid_res = None  # Call perform_grid_search
    results['Grid Search'] = None
    
    # Random search
    random_res = None  # Call perform_random_search
    results['Random Search'] = None
    
    # Bayesian optimization (if available)
    if OPTUNA_AVAILABLE:
        bayes_res = None  # Call perform_bayesian_optimization
        results['Bayesian Optimization'] = None
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Compare all tuning strategies
print("="*60)
print("COMPREHENSIVE STRATEGY COMPARISON")
print("="*60)

comparison_results = compare_tuning_strategies(X_train, y_train, n_iter_budget=50)

# Print summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"{'Method':<25} {'Best Score':<12} {'Time (s)':<12} {'Efficiency':<12}")
print("-"*60)

for method, results in comparison_results.items():
    efficiency = results['best_score'] / results['total_time']
    print(f"{method:<25} {results['best_score']:<12.4f} {results['total_time']:<12.2f} {efficiency:<12.6f}")

# Visualize comparison
compare_methods(comparison_results)

# Plot convergence curves
plt.figure(figsize=(12, 6))
for method, results in comparison_results.items():
    df = results['results_df']
    # Cumulative best score
    cumulative_best = np.maximum.accumulate(df['mean_score'].values)
    plt.plot(range(len(cumulative_best)), cumulative_best, 
             marker='o', linewidth=2, label=method, alpha=0.7)

plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Best Score Found', fontsize=12)
plt.title('Convergence Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)

best_score_method = max(comparison_results.items(), key=lambda x: x[1]['best_score'])
fastest_method = min(comparison_results.items(), key=lambda x: x[1]['total_time'])
most_efficient = max(comparison_results.items(), 
                    key=lambda x: x[1]['best_score'] / x[1]['total_time'])

print(f"\nBest Score: {best_score_method[0]} ({best_score_method[1]['best_score']:.4f})")
print(f"Fastest: {fastest_method[0]} ({fastest_method[1]['total_time']:.2f}s)")
print(f"Most Efficient: {most_efficient[0]} (score/sec: {most_efficient[1]['best_score']/most_efficient[1]['total_time']:.6f})")

print("\n Guidelines:")
print("  • Limited time? Use Random Search for quick results")
print("  • Expensive training? Use Bayesian Optimization for sample efficiency")
print("  • Few parameters? Grid Search for thorough exploration")
print("  • Production system? Combine multiple good configurations (ensemble)")

##### **Expected Output**
```
============================================================
COMPREHENSIVE STRATEGY COMPARISON
============================================================

============================================================
SUMMARY
============================================================
Method                    Best Score   Time (s)     Efficiency  
------------------------------------------------------------
Grid Search               0.9XXX       XX.XX        0.0XXXXX
Random Search             0.9XXX       XX.XX        0.0XXXXX
Bayesian Optimization     0.9XXX       XX.XX        0.0XXXXX

============================================================
RECOMMENDATIONS
============================================================

Best Score: XXX (0.9XXX)
Fastest: XXX (XX.XXs)
Most Efficient: XXX (score/sec: 0.0XXXXX)
```
Bayesian optimization typically achieves the best balance of score and efficiency, especially with limited iterations.

In [ ]:
unittests.exercise_5(compare_tuning_strategies)

<a name='7'></a>
## 7 - From-Scratch Implementation: Grid Search

Understanding how grid search works internally helps you customize it for specific needs.

### Core Components:

1. **Parameter Grid Generation**:
   - Create all combinations from parameter lists
   - Use itertools.product for Cartesian product

2. **Cross-Validation**:
   - Split data into K folds
   - For each fold: train on K-1, validate on 1
   - Average scores across folds

3. **Model Training**:
   - For each configuration: train model
   - Evaluate on validation fold
   - Record score

4. **Result Tracking**:
   - Store all configurations and scores
   - Track best configuration
   - Return detailed results

### Algorithm Pseudocode:

```
function GridSearch(X, y, param_grid, model_class, cv):
    results = []
    
    for params in generate_combinations(param_grid):
        scores = []
        
        for train_idx, val_idx in cv_splits(X, y, cv):
            model = model_class(**params)
            model.fit(X[train_idx], y[train_idx])
            score = model.score(X[val_idx], y[val_idx])
            scores.append(score)
        
        mean_score = mean(scores)
        results.append((params, mean_score))
    
    best_params = max(results, key=lambda x: x[1])[0]
    return best_params, results
```

<a name='ex-6'></a>
### Exercise 6 - Grid Search from Scratch

**Your Task:**

Implement `grid_search_scratch` function that:
1. Generates all parameter combinations
2. Implements K-fold cross-validation
3. Trains and evaluates each configuration
4. Returns best parameters and all results

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For parameter combinations:**
* Use itertools.product: `list(itertools.product(*param_grid.values()))`
* Create dict for each: `dict(zip(param_grid.keys(), combination))`

**For cross-validation splits:**
* Use sklearn.model_selection.KFold: `kf = KFold(n_splits=cv)`
* Iterate: `for train_idx, val_idx in kf.split(X):`

**For training:**
* Create model: `model = model_class(**params)`
* Train: `model.fit(X[train_idx], y[train_idx])`
* Evaluate: `score = model.score(X[val_idx], y[val_idx])`

**For results:**
* Store: params, mean_score, std_score for each configuration
* Best: `max(results, key=lambda x: x['mean_score'])`

</details>

In [ ]:
# GRADED FUNCTION: grid_search_scratch
def grid_search_scratch(X_train, y_train, param_grid, model_class, cv=3, scoring='accuracy'):
    """
    Implement grid search from scratch with cross-validation.
    
    Args:
        X_train: Training features
        y_train: Training labels
        param_grid: Dictionary with parameter names as keys and lists of values
        model_class: Model class to instantiate
        cv: Number of cross-validation folds
        scoring: Scoring metric ('accuracy' or callable)
    
    Returns:
        Dictionary with best_params, best_score, all_results
    """
    from itertools import product
    from sklearn.model_selection import KFold
    
    ### START CODE HERE ### (≈ 30 lines)
    
    # Generate all parameter combinations
    param_names = None
    param_values = None
    param_combinations = None  # Use itertools.product
    
    # Convert to list of dictionaries
    param_dicts = None
    
    # Initialize results storage
    all_results = []
    
    # Create KFold splitter
    kf = None
    
    # Iterate over all parameter combinations
    for params in param_dicts:
        # Store scores for this configuration
        fold_scores = []
        
        # Perform cross-validation
        for train_idx, val_idx in None:  # Use kf.split
            # Split data
            X_fold_train, X_fold_val = None, None
            y_fold_train, y_fold_val = None, None
            
            # Create and train model
            model = None  # Instantiate with params
            # Fit model
            
            # Evaluate
            if scoring == 'accuracy':
                score = None  # Use model.score or accuracy_score
            else:
                score = None  # Use custom scoring function
            
            fold_scores.append(score)
        
        # Compute mean and std
        mean_score = None
        std_score = None
        
        # Store results
        all_results.append({
            'params': params,
            'mean_score': mean_score,
            'std_score': std_score
        })
    
    # Find best configuration
    best_result = None  # max by mean_score
    best_params = None
    best_score = None
    
    ### END CODE HERE ###
    
    return {
        'best_params': best_params,
        'best_score': best_score,
        'all_results': all_results
    }

In [ ]:
# Test grid search from scratch
print("="*60)
print("GRID SEARCH FROM SCRATCH")
print("="*60)

# Define a small parameter grid for testing
test_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

# Run from-scratch implementation
start_time = time()
scratch_results = grid_search_scratch(
    X_train, y_train,
    param_grid=test_param_grid,
    model_class=RandomForestClassifier,
    cv=3,
    scoring='accuracy'
)
scratch_time = time() - start_time

print(f"\nCompleted in {scratch_time:.2f} seconds")
print(f"Best Score: {scratch_results['best_score']:.4f}")
print(f"Best Parameters: {scratch_results['best_params']}")
print(f"Total configurations: {len(scratch_results['all_results'])}")

# Compare with sklearn's GridSearchCV
print("\n" + "="*60)
print("COMPARISON WITH SKLEARN")
print("="*60)

start_time = time()
sklearn_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    test_param_grid,
    cv=3,
    scoring='accuracy'
)
sklearn_grid.fit(X_train, y_train)
sklearn_time = time() - start_time

print(f"\nSklearn GridSearchCV:")
print(f"  Time: {sklearn_time:.2f}s")
print(f"  Best Score: {sklearn_grid.best_score_:.4f}")
print(f"  Best Params: {sklearn_grid.best_params_}")

print(f"\nFrom Scratch:")
print(f"  Time: {scratch_time:.2f}s")
print(f"  Best Score: {scratch_results['best_score']:.4f}")
print(f"  Best Params: {scratch_results['best_params']}")

print(f"\nScore difference: {abs(sklearn_grid.best_score_ - scratch_results['best_score']):.6f}")
print("(Should be very close to 0)")

# Visualize results
results_df = pd.DataFrame(scratch_results['all_results'])
results_df = results_df.sort_values('mean_score', ascending=False).reset_index(drop=True)

print(f"\nTop 5 configurations:")
print(results_df.head())

plt.figure(figsize=(12, 6))
plt.errorbar(range(len(results_df)), results_df['mean_score'], 
             yerr=results_df['std_score'], marker='o', linewidth=2,
             capsize=4, capthick=2, alpha=0.7)
plt.axhline(y=scratch_results['best_score'], color='red', linestyle='--',
           label=f'Best: {scratch_results["best_score"]:.4f}')
plt.xlabel('Configuration', fontsize=12)
plt.ylabel('Mean CV Score', fontsize=12)
plt.title('Grid Search from Scratch - All Configurations', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

##### **Expected Output**
```
============================================================
GRID SEARCH FROM SCRATCH
============================================================

Completed in XX.XX seconds
Best Score: 0.9XXX
Best Parameters: {'max_depth': XX, 'min_samples_split': X, 'n_estimators': XXX}
Total configurations: 12

============================================================
COMPARISON WITH SKLEARN
============================================================

Sklearn GridSearchCV:
  Time: XX.XXs
  Best Score: 0.9XXX
  Best Params: {'max_depth': XX, ...}

From Scratch:
  Time: XX.XXs
  Best Score: 0.9XXX
  Best Params: {'max_depth': XX, ...}

Score difference: 0.000000
(Should be very close to 0)
```
Your implementation should produce nearly identical results to sklearn's GridSearchCV, validating correctness.

In [ ]:
unittests.exercise_6(grid_search_scratch)

<a name='8'></a>
## 8 - Conclusion

**Congratulations!** You've completed the Hyperparameter Tuning assignment!

### What You've Learned:

1. **Grid Search**: Exhaustive search over parameter combinations with guaranteed coverage
2. **Random Search**: Efficient random sampling for faster exploration
3. **Bayesian Optimization**: Intelligent sequential search using probabilistic models
4. **Parameter Importance**: Identifying which hyperparameters matter most
5. **Strategy Comparison**: Understanding tradeoffs between different approaches
6. **From-Scratch Implementation**: Building grid search with cross-validation

### Key Takeaways:

**Method Selection:**
- **Grid Search**: Few parameters (2-3), need thorough exploration
- **Random Search**: Many parameters, quick first pass, good baseline
- **Bayesian Optimization**: Limited budget, expensive training, sample efficiency

**Best Practices:**
- Start with default parameters, establish baseline
- Use random search for initial exploration (50-100 configs)
- Analyze parameter importance to focus efforts
- Use Bayesian optimization for refined search
- Always use cross-validation, never tune on test set
- Consider early stopping for efficiency
- Document best configurations for reproducibility
- Ensemble multiple good configurations

**Common Pitfalls:**

1. **Overfitting to Validation**: Too many trials can overfit to validation set
   - Solution: Use nested cross-validation or separate validation set

2. **Ignoring Computational Cost**: Not all configurations are equally expensive
   - Solution: Use time-based budgets, early stopping

3. **Too Fine Granularity**: Excessive grid resolution wastes time
   - Solution: Start coarse, then refine promising regions

4. **Ignoring Interactions**: Parameters often interact non-trivially
   - Solution: Consider joint optimization, analyze interactions

5. **Wrong Metric**: Optimizing wrong objective
   - Solution: Align tuning metric with business goal

### Advanced Topics:

**Multi-Objective Optimization:**
- Pareto frontier: accuracy vs model size
- Optimize accuracy, inference time, memory simultaneously

**Multi-Fidelity Optimization:**
- Hyperband: Allocate resources adaptively
- BOHB: Bayesian optimization + Hyperband
- Train on subset, evaluate promising ones on full data

**Neural Architecture Search (NAS):**
- Automated architecture design
- DARTS, ENAS, NAS-Bench

**Meta-Learning:**
- Transfer hyperparameters across tasks
- Learn initialization strategies

**Automated Machine Learning (AutoML):**
- Auto-sklearn, TPOT, AutoKeras
- End-to-end pipeline optimization

### Real-World Applications:

**Production Systems:**
- Continuous retraining with hyperparameter updates
- A/B testing different configurations
- Model registry with performance tracking

**Resource-Constrained:**
- Edge devices: optimize for size/speed
- Mobile: balance accuracy and battery
- Cloud: optimize cost per prediction

**Competition ML:**
- Extensive tuning for marginal gains
- Ensemble of diverse configurations
- Stacking tuned models

### Tools and Libraries:

**Hyperparameter Optimization:**
- Optuna: Modern, flexible, great visualizations
- Hyperopt: TPE algorithm, mature
- Scikit-Optimize: Bayesian optimization, sklearn integration
- Ray Tune: Distributed, scalable, multi-fidelity

**AutoML Frameworks:**
- Auto-sklearn: Automated sklearn pipelines
- TPOT: Genetic programming for pipelines
- H2O AutoML: End-to-end automation
- AutoKeras: Neural architecture search

**Experiment Tracking:**
- MLflow: Experiment tracking, model registry
- Weights & Biases: Real-time monitoring
- Neptune.ai: Metadata management

### Recommended Resources:

- **Papers**: 
  - "Random Search for Hyperparameter Optimization" (Bergstra & Bengio, 2012)
  - "Practical Bayesian Optimization" (Snoek et al., 2012)
  - "Hyperband" (Li et al., 2018)

- **Books**: 
  - "Automated Machine Learning" (Hutter et al., 2019)
  - "Bayesian Optimization" (Garnett, 2023)

- **Courses**: 
  - AutoML course (University of Freiburg)
  - Hyperparameter tuning on Coursera

**Excellent work! You now have the skills to optimize any machine learning model efficiently!**